# Урок 1.1 — RAW в S3 (MinIO): загрузка батча с `ingest_date`

Этот ноутбук — вариант **Пути B** из урока 1.1: мы берём CSV из `/data/csv` и складываем их в MinIO (S3) в бакет `raw`.

**Как работать:**
1) Поменяй `ingest_date` (ячейка 3).
2) Запусти ячейки сверху вниз.
3) Проверь, что в MinIO появились папки вида `raw/olist/<dataset>/ingest_date=.../`.


## 1) SparkSession

> Если в выводе Spark написано `Master = local[*]` — это нормально для этого урока.
В модуле по Spark-кластеру мы отдельно разберём, как подключаться к `spark-master`.


In [ ]:
import os, glob
from pyspark.sql import SparkSession

# 1) Собираем JAR'ы (S3 + Postgres) из /jars
jars = sorted(glob.glob("/jars/*.jar"))
assert any("hadoop-aws" in j for j in jars), f"hadoop-aws jar not found in /jars: {jars}"
assert any("aws-java-sdk-bundle" in j for j in jars), f"aws sdk jar not found in /jars: {jars}"
assert any("postgresql" in j for j in jars), f"postgresql jar not found in /jars: {jars}"
print("Using jars:", jars)

# 2) Выбираем режим:
# - для простых уроков оставь local[*]
# - для кластера включи SPARK_MASTER в env (у тебя он уже есть) и возьми его
master = os.getenv("SPARK_MASTER", "local[*]")  # например: spark://spark-master:7077
print("Spark master =", master)

# 3) Гасим старый контекст (иначе новые конфиги/JAR не применятся)
try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("lesson01_io")
    .master(master)
    # JAR'ы
    .config("spark.jars", ",".join(jars))
    .config("spark.driver.extraClassPath", ":".join(jars))
    .config("spark.executor.extraClassPath", ":".join(jars))

    # Если запускаешь на кластере из контейнера jupyter — полезно явно зафиксировать driver host
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.host", "jupyter")

    # Чтобы не словить "no resources" на маленьких воркерах:
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "2")
    .config("spark.executor.memory", "512m")
    .config("spark.executor.memoryOverhead", "256m")

    # S3A (MinIO)
    .config("spark.hadoop.fs.s3a.endpoint", os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ACCESS_KEY", "airflow"))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_SECRET_KEY", "airflow123"))
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)

# 4) Быстрый тест: класс обязан существовать
spark._jvm.java.lang.Class.forName("org.apache.hadoop.fs.s3a.S3AFileSystem")
print("S3AFileSystem is OK ✅")

print("sparkContext.master =", spark.sparkContext.master)


## 2) Параметры батча

- `src_base` — где лежат исходные CSV (внутри контейнера).
- `dst_base` — куда пишем RAW в S3 (MinIO).

> В MinIO ты увидишь это как: бакет `raw` → префикс `olist/...`.


In [ ]:
ingest_date = "2026-01-04"       # выбери свою дату
src_base = "/data/csv"
dst_base = "s3a://raw/olist"


## 3) Список датасетов

Тут мы задаём соответствие: `папка_в_raw` → `имя_csv`.


In [ ]:
datasets = [
    ("orders", "olist_orders_dataset.csv"),
    ("order_items", "olist_order_items_dataset.csv"),
    ("customers", "olist_customers_dataset.csv"),
    ("order_payments", "olist_order_payments_dataset.csv"),
    ("products", "olist_products_dataset.csv"),
    ("sellers", "olist_sellers_dataset.csv"),
    ("reviews", "olist_order_reviews_dataset.csv"),
    ("category_translation", "product_category_name_translation.csv"),
    # если у тебя есть geolocation как CSV — добавь сюда:
    # ("geolocation", "olist_geolocation_dataset.csv"),
]


## 4) Загрузка батча в RAW (S3)

Ключевые идеи:
- **RAW не типизируем** (`inferSchema=False`).
- Пишем в папку `ingest_date=...`.
- Для идемпотентности на одну `ingest_date` делаем `mode("overwrite")`.
- `coalesce(1)` нужен, чтобы получить **один** part-файл (удобнее смотреть руками в MinIO).

> В S3 Spark всё равно пишет **part-*.csv** внутри папки ingest_date — это нормально.


In [ ]:
for folder, filename in datasets:
    src_path = f"{src_base}/{filename}"
    dst_path = f"{dst_base}/{folder}/ingest_date={ingest_date}/"

    # Читаем исходный CSV как строки (RAW): без попытки угадать типы
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)
          .csv(src_path))

    # Идемпотентность: на одну ingest_date перезаписываем папку батча целиком
    (df.coalesce(1)  # 1 part-файл на датасет (чтобы руками было проще проверять)
       .write
       .mode("overwrite")
       .option("header", True)
       .csv(dst_path))

    print(f"✅ {filename} -> {dst_path}")


## 5) Быстрая проверка

Проверим, что файлы действительно читаются из `s3a://...`.
Если в MinIO ты видишь только «папку», просто провались внутрь — там будут `part-*.csv` и служебные файлы.


In [ ]:
# Пример проверки на customers (можешь поменять на любой другой датасет)
check_path = f"{dst_base}/customers/ingest_date={ingest_date}/"

df_check = (spark.read
            .option("header", True)
            .csv(check_path))

print("files sample:", df_check.inputFiles()[:5])
print("rows:", df_check.count())
df_check.show(5, truncate=False)
